# Cat Breed Classification

The project has a character generation from photo and description feature. It consists of 4 step - cat breed classification, color extraction, character card generation, avatar generation. The development and engineering of the first step will be covered in this notebook. 

## Setup

In [ ]:
import os
from transformers import pipeline
from PIL import Image
import io
import json
import requests

To start with, cat breed classification is a solved problem. There are many already trained models for this purpose. For this case, a model from hugging face will be used. 
- Cat breed image detection ViT (https://huggingface.co/dima806/cat_breed_image_detection)

The model is a finetuned version of google/vit-base-patch16-224-in21k, trained on a combination of 2 datasets from kaggle:
- Cat Breeds Dataset (https://www.kaggle.com/datasets/ma7555/cat-breeds-dataset)
- CatBreedsRefined-7k (https://www.kaggle.com/datasets/doctrinek/catbreedsrefined-7k)

During training the model produced stable results, with high precision and recall for most cat breeds. Accuracy of the model - 77%.  

In [44]:
# pipeline already does all the steps needed for classifications, therefore creating a separate pipeline for data preprocessing is not neccessary 
classifier = pipeline("image-classification", model="dima806/cat_breed_image_detection")

result = classifier("../resources/sphynx.jpeg")
result


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[{'label': 'Sphynx', 'score': 0.9989038705825806},
 {'label': 'Tonkinese', 'score': 5.5279353546211496e-05},
 {'label': 'Abyssinian', 'score': 5.033482739236206e-05},
 {'label': 'Cornish Rex', 'score': 4.431399793247692e-05},
 {'label': 'Burmese', 'score': 4.394847564981319e-05}]

In [45]:
def classify(url: str) -> str:
    classifier = pipeline("image-classification", model="dima806/cat_breed_image_detection")
    return classifier(url)

In [46]:
print(classify("../resources/sphynx.jpeg"))

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[{'label': 'Sphynx', 'score': 0.9989038705825806}, {'label': 'Tonkinese', 'score': 5.5279353546211496e-05}, {'label': 'Abyssinian', 'score': 5.033482739236206e-05}, {'label': 'Cornish Rex', 'score': 4.431399793247692e-05}, {'label': 'Burmese', 'score': 4.394847564981319e-05}]


The output of the model is a list of objects. For future breed name retrieval the following code would be used

In [47]:
result[0]["label"]

'Sphynx'

Since this classification would be used as a part of game's backend API, and the image would be sent over as raw bytes, the response has to be transformed into an image. To test the behavior, I will use the Cat API (https://thecatapi.com)

In [ ]:
api_key= os.get_env(CAT_API_KEY)
x = requests.get(f'https://api.thecatapi.com/v1/images/search?breed_ids=abys&api_key={api_key}')

data = x.json()

url = data[0]["url"]
print(url)

https://cdn2.thecatapi.com/images/9x1zk_Qdr.jpg


In [49]:
def classify_breed(image_bytes: bytes) -> str:
    image = Image.open(io.BytesIO(image_bytes))
    results = classifier(image)
    return results[0]["label"]

In [ ]:
response = requests.get(url)
cat_to_classify = response.content
classify_breed(cat_to_classify)

'Abyssinian'

The classification pipeline works, and is ready for implementation. 